In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
# to avoid API failures

def generate_completion(model, system_prompt, user_content):
    
    """Helper to handle API calls with basic retry logic"""
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],

                # for stable reasoning
                temperature=0.0
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2) # Wait 2s before retry
            else:
                print(f"  [!] Failed after {max_retries} attempts: {e}")
                return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# SROIE (Full Training Set)
INPUT_FILE = "phase1_sroie_input_train.jsonl"
OUTPUT_FILE = "phase1b_sroie_train_reasoning.jsonl"
MODEL = "gpt-4.1"

SYSTEM_PROMPT = """You are an Expert Accountant Agent.
Task: Extract structured data from the receipt OCR text provided.
Output Format:
Thought: <Explain step-by-step where you found the Date, Total, and Merchant.>
JSON: <The final JSON object with keys: merchant, date, total, address>
"""


In [5]:
print(f"Starting SROIE Processing ({INPUT_FILE})...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} receipts to process.")
        
        for i, line in enumerate(lines):

            # read receipts
            data = json.loads(line)

            # teacher inference
            output = generate_completion(MODEL, SYSTEM_PROMPT, data['input'])
            
            if output:

                # saving reasoning
                data['output'] = output
                f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 20 == 0: 
                print(f"  [SROIE] Processed {i+1}/{len(lines)}")

    print(f"SROIE Complete! Saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Starting SROIE Processing (phase1_sroie_input_train.jsonl)...
  -> Found 709 receipts to process.
  [SROIE] Processed 20/709
  [SROIE] Processed 40/709
  [SROIE] Processed 60/709
  [SROIE] Processed 80/709
  [SROIE] Processed 100/709
  [SROIE] Processed 120/709
  [SROIE] Processed 140/709
  [SROIE] Processed 160/709
  [SROIE] Processed 180/709
  [SROIE] Processed 200/709
  [SROIE] Processed 220/709
  [SROIE] Processed 240/709
  [SROIE] Processed 260/709
  [SROIE] Processed 280/709
  [SROIE] Processed 300/709
  [SROIE] Processed 320/709
  [SROIE] Processed 340/709
  [SROIE] Processed 360/709
  [SROIE] Processed 380/709
  [SROIE] Processed 400/709
  [SROIE] Processed 420/709
  [SROIE] Processed 440/709
  [SROIE] Processed 460/709
  [SROIE] Processed 480/709
  [SROIE] Processed 500/709
  [SROIE] Processed 520/709
  [SROIE] Processed 540/709
  [SROIE] Processed 560/709
  [SROIE] Processed 580/709
  [SROIE] Processed 600/709
  [SROIE] Processed 620/709
  [SROIE] Processed 640/709
  [SROIE] 